In [ ]:
from unsloth import FastLanguageModel
import torch

fourbit_models = [
    "unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit", # Qwen 14B 2x faster
    "unsloth/Qwen3-4B-Thinking-2507-unsloth-bnb-4bit",
    "unsloth/Qwen3-8B-unsloth-bnb-4bit",
    "unsloth/Qwen3-14B-unsloth-bnb-4bit",
    "unsloth/Qwen3-32B-unsloth-bnb-4bit",

    # 4bit dynamic quants for superior accuracy and low memory use
    "unsloth/gemma-3-12b-it-unsloth-bnb-4bit",
    "unsloth/Phi-4",
    "unsloth/Llama-3.1-8B",
    "unsloth/Llama-3.2-3B",
    "unsloth/orpheus-3b-0.1-ft-unsloth-bnb-4bit" # [NEW] We support TTS models!
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-4B-Instruct-2507",
    max_seq_length = 2048, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    load_in_8bit = False, # [NEW!] A bit more accurate, uses 2x memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

In [ ]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen3-instruct",
)

In [2]:
from datasets import load_dataset

dataset = load_dataset(
    "json", 
    data_files=r"D:\Code\pdf-workbench\pdf_mining\training_data\outline\three_column\sft_labeled_2026_02_25_16_07_40.jsonl",
    split="train"  # 如果不加这个，返回的是 DatasetDict，加了直接返回 Dataset
)

Generating train split: 0 examples [00:00, ? examples/s]

In [3]:
dataset[0]

{'conversations': [{'content': '# 任务描述\n你是一个专业的文档分析助手，负责从OCR提取的文本中识别真正的文档提纲内容，并确定其层级关系。这是一个**提取任务**，你需要从给定的候选内容中提取出真正属于提纲的标题部分，并按照指定格式输出。\n\n# 提纲判断标准\n## 1. 典型的提纲标题形式\n以下形式的内容**可能**属于提纲：\n- 以数字序号开头：如"1."、"2、"、"3．"、"1.1"、"1.1.1"\n- 以中文数字序号开头：如"一、"、"二、"、"（一）"、"（二）"、"第一章"、"第一节"\n- 以字母序号开头：如"A."、"B、"、"（A）"、"（B）"\n- 以特定词语开头：如"附录"、"附表"、"附件"、"附则"、"摘要"、"Abstract"、"引言"、"结论"、"参考文献"、"致谢"等文档标准结构部分\n- 带有明显的层级标记：如"§1"、"§2"、"第1条"、"第2条"\n\n## 2. 提纲内容特征\n- 提纲标题通常较短（一般不超过50个字符）\n- 提纲标题后通常有换行或明显的内容分隔\n- 提纲标题往往概括性较强，而不是具体的论述内容\n- **排除整个文档的主标题**（如论文标题、报告标题、文件名称等）\n- **包含文档的标准结构部分**（如摘要、引言、结论、参考文献等，即使没有数字序号也应提取）\n\n## 3. 需要排除的非提纲内容\n以下情况**不应**作为提纲提取：\n- 整个文档的主标题、论文标题、报告标题、文件名称等\n- 虽然以序号开头，但实际上是正文内容（如"1. 我们知道，在历史上..."）\n- 序号后跟着详细说明或冒号后的具体内容（只提取提纲标题部分，去除后续说明）\n- 目录页内容（当页面出现"目录"字眼、且内容为"标题+页码"格式时，跳过整个页面）\n- 页眉、页脚、页码等非正文内容\n- 表格内的序号内容（除非整个表格是提纲列表）\n- 四级、五级等更深层级的提纲标题（只处理一、二、三级提纲）\n\n## 4. 提取规则\n- **提取完整标题**：提取从序号或特定标题词开始，直到标题核心语义结束的部分\n  - 如果原始内容有序号，必须包含完整序号（如"3.2 预算安排"）\n  - 如果以"附录"、"附件"、"摘要"等特定词语开头，包含该词语及后续标题（如

In [4]:
def formatting_prompts_func(examples):
   convos = examples["conversations"]
   texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
   return { "text" : texts, }

dataset = dataset.map(formatting_prompts_func, batched = True)

Map:   0%|          | 0/104 [00:00<?, ? examples/s]

NameError: name 'tokenizer' is not defined

In [5]:
dataset[0]['text']

KeyError: 'text'

In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset = None, # Can set up evaluation!
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4, # Use GA to mimic batch size!
        warmup_steps = 5,
        # num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 60,
        learning_rate = 2e-4, # Reduce to 2e-5 for long training runs
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

We also use Unsloth's `train_on_completions` method to only train on the assistant outputs and ignore the loss on the user's inputs. This helps increase accuracy of finetunes!

In [6]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

ModuleNotFoundError: No module named 'unsloth'

Let's verify masking the instruction part is done! Let's print the 0th row again.

In [7]:
tokenizer.decode(trainer.train_dataset[100]["input_ids"])

NameError: name 'tokenizer' is not defined

Now let's print the masked out example - you should see only the answer is present:

In [ ]:
tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[100]["labels"]]).replace(tokenizer.pad_token, " ")

Show current memory stats

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

Let's train the model! To resume a training run, set `trainer.train(resume_from_checkpoint = True)`

In [ ]:
trainer_stats = trainer.train()

Show final memory and time stats

In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")